# 4. Using existing flows: MgO Phonons

In this section, we make use of density functional **perturbation** theory (DFPT) for computing the phonon band structure of magnesium oxide.


In [2]:
from pathlib import Path
import shutil
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from abipy import abilab
import abipy.flowtk as flowtk
abilab.enable_notebook()

%matplotlib inline

import workshop_lib as wlib

## 3.1 Phonons from DFPT

Density-functional perturbation theory (DFPT) gives access to phonons,
Born effective charges and the dielectric tensor without finite
displacements. AbiPy's `PhononFlow` automates the whole workflow: a GS run
that produces the ground-state wavefunctions (`WFK`), followed by one DFPT
task per symmetry-irreducible atomic perturbation and q-point.

`wlib.build_mgo_phonon_flow` builds a simplified, MgO version of this --
coarser `ecut`, k-mesh and q-mesh, for speed -- and has already been run
for you (`flow_mgo_phonons/`, next to this notebook). Standalone version:
`../Examples/make_mgo_phonons.py`. MgO (rocksalt) is strongly ionic, which
makes for a good showcase of the LO-TO splitting below.


In [3]:
wlib.print_source(wlib.build_mgo_phonon_flow)

### Build and run the flow

Standalone version: `../Examples/make_mgo_phonons.py`. This one has more
tasks than the others (one per symmetry-irreducible atomic perturbation
and q-point), so it's the best candidate for actually running with
`nohup` rather than waiting on it in a foreground shell:

```
cd ../Examples
nohup python make_mgo_phonons.py > log 2> err &
abirun.py flow_mgo_phonons scheduler
```

This has already been run for you (`flow_mgo_phonons/`, next to this
notebook) -- the analysis below reads directly from it.


All the DFPT results (dynamical matrices, Born effective charges,
dielectric tensor) are merged into a single `DDB` file -- the entry point
for essentially all the post-processing below:


In [ ]:
ddb = abilab.abiopen("flow_mgo_phonons/w1/outdata/out_DDB")
print(ddb)

`ddb.anaget_phbst_and_phdos_files` calls `anaddb` behind the scenes to
Fourier-interpolate the dynamical matrix onto a dense q-mesh (for the
phonon DOS) and along a high-symmetry q-path (for the phonon band
structure) -- this is exactly what `../Examples/run_mgo_anaddb.py` does,
standalone:


In [ ]:
phbst_file, phdos_file = ddb.anaget_phbst_and_phdos_files(
    nqsmall=10, ndivsm=10, asr=2, chneut=1, dipdip=1,
    dos_method="tetra", lo_to_splitting=True)

phbands = phbst_file.phbands
phbands.plot_with_phdos(phdos_file.phdos);

> **Note.** `lo_to_splitting=True` accounts for the LO-TO splitting at
> $\Gamma$ that polar materials like MgO show, using the Born effective
> charges and the dielectric tensor also stored in the `DDB`.

Born effective charges and the dielectric tensor can be inspected directly:


In [ ]:
epsinf, becs = ddb.anaget_epsinf_and_becs()
print("Electronic dielectric tensor:\n", epsinf)
print("\nBorn effective charges:")
print(becs)

Keep `phdos_file` open for now -- `4-Assignment.ipynb` uses it for a
harmonic-thermodynamics exercise. Otherwise, remember to close what you
open:


In [ ]:
phbst_file.close()
ddb.close()